In [ ]:
import random
import math
class SimulatedAnnealing:
    def __init__(self,container,intitial_boxes,T=1000,alpha=0.95):
        self.container = container
        self.current = intitial_boxes
        self.T = T
        self.alpha = alpha

    def get_neighbor(self, solution):
        neighbor = solution.copy()

        i, j = random.sample(range(len(neighbor)), 2)
        neighbor[i], neighbor[j] = neighbor[j], neighbor[i]

        return neighbor
    
    def evaluate(self, solution):
       

        result = self.container.pack_boxes(solution)

        utilization = result["utilization"]
        stability = result["stability"]["stability_score"]

        w_util = 0.7
        w_stab = 0.3

        return w_util * utilization + w_stab * stability
    

    def accept(self, current_score, new_score,T):
    
        # if new solution is better → always accept
        if new_score > current_score:
            return True
        T = max(T, 1e-10)
        # if worse → accept with probability
        delta = current_score - new_score
        probability = math.exp(-delta / T)

        return random.random() < probability
    

    def run(self,iterations=1000):

        # ── 1. INITIALIZATION ─────────────────────────────
        current_solution = self.current
        current_score = self.evaluate(current_solution)

        best_solution = current_solution
        best_score = current_score

        T = self.T   # initial temperature

        # ── 2. MAIN LOOP ───────────────────────────────────
        for _ in range(iterations):

            # generate neighbor
            neighbor = self.get_neighbor(current_solution)
            neighbor_score = self.evaluate(neighbor)

            # decide accept/reject
            if self.accept(current_score, neighbor_score,T):

                current_solution = neighbor
                current_score = neighbor_score

                # update best solution
                if neighbor_score > best_score:
                    best_solution = neighbor
                    best_score = neighbor_score

            # cool down
            T *= self.alpha
            

        # ── 3. RETURN RESULT ───────────────────────────────
        return {
            "best_solution": best_solution,
            "best_score": best_score,
            "best_result": self.container.pack_boxes(best_solution)  # final state
        }